In [1]:
!pip install matplotlib seaborn torch pandas numpy scikit-learn ucimlrepo

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo
import os
import glob # For finding files


# --- Configuration ---
DATASET_ID = 442
# DEVICE_NAME = "Ecobee_Thermostat" # Or any of the 9 devices
# DEVICE_NAMES = ["Danmini_Doorbell", "Ecobee_Thermostat", "Ennio_Doorbell"]
DEVICE_NAMES = ["Danmini_Doorbell", "Ecobee_Thermostat", "Ennio_Doorbell", "Philips_B120N10_Baby_Monitor", "Provision_PT_737E_Security_Camera", "Provision_PT_838_Security_Camera", "Samsung_SNH_1011_N_Webcam", "SimpleHome_XCS7_1002_WHT_Security_Camera", "SimpleHome_XCS7_1003_WHT_Security_Camera"]
BASE_DATA_PATH = "./data" # CHANGE THIS to your extracted dataset path
# Ensure this base path exists, or adapt to how ucimlrepo stores data if using it directly
# For ucimlrepo, it might download to a specific cache location or allow specification.

# --- Fetch and/or Locate Data ---
try:
    # Check if data is already downloaded/extracted by user
    full_df = pd.DataFrame([])
    for DEVICE_NAME in DEVICE_NAMES:
        device_path = os.path.join(BASE_DATA_PATH, DEVICE_NAME)
        if not os.path.exists(device_path):
            print(f"Device data for {DEVICE_NAME} not found at {device_path}.")
            print("Attempting to fetch using ucimlrepo (this might download the whole dataset)...")
            if not os.path.exists(BASE_DATA_PATH):
                os.makedirs(BASE_DATA_PATH)

            iot_botnet_attacks = fetch_ucirepo(id=DATASET_ID)
            # ucimlrepo fetch_ucirepo typically loads data into memory (X, y)
            # and provides metadata. It doesn't directly extract into a file structure
            # like the one N-BaIoT comes in (per-device CSVs).
            # So, for N-BaIoT, manual download and extraction is usually preferred
            # to work with individual device files easily.

            # If you manually downloaded and extracted the ZIP from UCI:
            # The structure is usually:
            # archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip
            # Unzipping this gives you a folder, and inside that, folders for each of the 9 devices.
            # e.g., ./N-BaIoT/1. Danmini_Doorbell/benign_traffic.csv
            # e.g., ./N-BaIoT/1. Danmini_Doorbell/gafgyt_attacks/combo.csv
            # You might need to adjust DEVICE_NAME and paths if your extraction created numbered folders.
            # For now, let's assume you've placed the Danmini_Doorbell files directly under device_path.
            # Example:
            # device_path = "./N-BaIoT_data/Danmini_Doorbell/"
            # benign_file = os.path.join(device_path, "benign_traffic.csv")
            # attack_files_pattern = os.path.join(device_path, "gafgyt_attacks/*.csv") # For Gafgyt
            # attack_files_pattern_mirai = os.path.join(device_path, "mirai_attacks/*.csv") # For Mirai
            print(f"Please ensure you have manually downloaded and extracted the dataset from")
            print(f"https://archive.ics.uci.edu/static/public/442/detection+of+iot+botnet+attacks+n+baiot.zip")
            print(f"into a structure like {BASE_DATA_PATH}/{DEVICE_NAME}/")
            raise FileNotFoundError("Dataset files not found in expected local path.")

        print(f"Using data from: {device_path}")

        # Load benign traffic
        benign_file_path = os.path.join(device_path, "benign_traffic.csv")
        if not os.path.exists(benign_file_path):
            raise FileNotFoundError(f"Benign traffic file not found: {benign_file_path}")
        benign_df = pd.read_csv(benign_file_path)
        benign_df['label'] = 0 # 0 for benign

        # Load attack traffic
        # We'll combine all attacks for that device into a single "attack" class
        attack_dfs = []
        attack_folders = [
            os.path.join(device_path, "gafgyt_attacks"),
            os.path.join(device_path, "mirai_attacks")
        ]

        for folder in attack_folders:
            if os.path.exists(folder):
                # The N-BaIoT dataset description mentions 10 attacks for each device.
                # These are often individual CSV files within the 'gafgyt_attacks' and 'mirai_attacks' subfolders.
                # Example: gafgyt_attacks/combo.csv, gafgyt_attacks/junk.csv, etc.
                # mirai_attacks/ack.csv, mirai_attacks/scan.csv, etc.
                csv_files = glob.glob(os.path.join(folder, "*.csv"))
                if not csv_files:
                    print(f"Warning: No CSV files found in attack folder: {folder}")
                for attack_file in csv_files:
                    try:
                        print(f"Loading attack file: {attack_file}")
                        attack_df_single = pd.read_csv(attack_file)
                        attack_dfs.append(attack_df_single)
                    except Exception as e:
                        print(f"Error loading {attack_file}: {e}")
            else:
                print(f"Warning: Attack folder not found: {folder}")

        if not attack_dfs:
            raise FileNotFoundError(f"No attack traffic files loaded for {DEVICE_NAME}. Check paths and file structure.")

        attack_df_combined = pd.concat(attack_dfs, ignore_index=True)
        attack_df_combined['label'] = 1 # 1 for attack

        # Combine benign and attack data
        full_df = pd.concat([full_df, benign_df, attack_df_combined], ignore_index=True)

    print(f"Full dataset shape: {full_df.shape}")
    print(f"Class distribution:\n{full_df['label'].value_counts()}")

    # Separate features and labels
    X = full_df.drop('label', axis=1).values
    y = full_df['label'].values

    # Data splitting (train, validation, test)
    # First, split into training and temp (validation + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y # stratify is important
    )
    # Then, split temp into validation and test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    print(f"Training set shape: {X_train.shape}, {y_train.shape}")
    print(f"Validation set shape: {X_val.shape}, {y_val.shape}")
    print(f"Test set shape: {X_test.shape}, {y_test.shape}")

    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val) # Use transform, not fit_transform
    X_test = scaler.transform(X_test) # Use transform, not fit_transform

    print("Data loading and preprocessing complete.")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please ensure the N-BaIoT dataset is downloaded, extracted, and the BASE_DATA_PATH is set correctly.")
    print("The expected structure for a device (e.g., Danmini_Doorbell) is:")
    print(f"{BASE_DATA_PATH}/{DEVICE_NAME}/benign_traffic.csv")
    print(f"{BASE_DATA_PATH}/{DEVICE_NAME}/gafgyt_attacks/*.csv")
    print(f"{BASE_DATA_PATH}/{DEVICE_NAME}/mirai_attacks/*.csv")
    # Exit if data can't be loaded
    exit()
except Exception as e:
    print(f"An unexpected error occurred during data preparation: {e}")
    exit()

Using data from: ./data\Danmini_Doorbell
Loading attack file: ./data\Danmini_Doorbell\gafgyt_attacks\combo.csv
Loading attack file: ./data\Danmini_Doorbell\gafgyt_attacks\junk.csv
Loading attack file: ./data\Danmini_Doorbell\gafgyt_attacks\scan.csv
Loading attack file: ./data\Danmini_Doorbell\gafgyt_attacks\tcp.csv
Loading attack file: ./data\Danmini_Doorbell\gafgyt_attacks\udp.csv
Loading attack file: ./data\Danmini_Doorbell\mirai_attacks\ack.csv
Loading attack file: ./data\Danmini_Doorbell\mirai_attacks\scan.csv
Loading attack file: ./data\Danmini_Doorbell\mirai_attacks\syn.csv
Loading attack file: ./data\Danmini_Doorbell\mirai_attacks\udp.csv
Loading attack file: ./data\Danmini_Doorbell\mirai_attacks\udpplain.csv
Using data from: ./data\Ecobee_Thermostat
Loading attack file: ./data\Ecobee_Thermostat\gafgyt_attacks\combo.csv
Loading attack file: ./data\Ecobee_Thermostat\gafgyt_attacks\junk.csv
Loading attack file: ./data\Ecobee_Thermostat\gafgyt_attacks\scan.csv
Loading attack file: 

In [3]:
# This script assumes 'full_df' is available from the previous data loading steps.

print("--- Exporting Segregated Feature Distribution Summaries ---")

# Ensure 'full_df' exists and has the 'label' column
if 'full_df' in locals() and 'label' in full_df.columns:
    # 1. Separate the DataFrame into benign (label=0) and attack (label=1)
    benign_df = full_df[full_df['label'] == 0].drop('label', axis=1)
    attack_df = full_df[full_df['label'] == 1].drop('label', axis=1)

    # --- Process and Export Benign Data ---
    if not benign_df.empty:
        # 2. Calculate descriptive statistics for benign features
        benign_stats = benign_df.describe().transpose()

        # 3. Export the benign statistics to a CSV file
        benign_output_filename = 'feature_distribution_summary_benign.csv'
        benign_stats.to_csv(benign_output_filename)
        print(f"\n✅ Successfully exported benign feature distribution.")
        print(f"   -> Find it in the file: '{benign_output_filename}'")
    else:
        print("\n⚠️ Warning: No benign data found to export.")

    # --- Process and Export Attack Data ---
    if not attack_df.empty:
        # 2. Calculate descriptive statistics for attack features
        attack_stats = attack_df.describe().transpose()

        # 3. Export the attack statistics to a CSV file
        attack_output_filename = 'feature_distribution_summary_attack.csv'
        attack_stats.to_csv(attack_output_filename)
        print(f"\n✅ Successfully exported attack feature distribution.")
        print(f"   -> Find it in the file: '{attack_output_filename}'")
    else:
        print("\n⚠️ Warning: No attack data found to export.")

else:
    print("\n❌ Error: 'full_df' DataFrame not found or is missing the 'label' column.")
    print("Please ensure the initial data loading script has been run successfully.")

--- Exporting Segregated Feature Distribution Summaries ---

✅ Successfully exported benign feature distribution.
   -> Find it in the file: 'feature_distribution_summary_benign.csv'

✅ Successfully exported attack feature distribution.
   -> Find it in the file: 'feature_distribution_summary_attack.csv'
